# Solutions — Beans, Getters/Setters, Optional, Builders, Static Utils → Python

Exercise notebook: `04_java_to_python/03_classes_properties_optional.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · A Java Bean → @dataclass

**Working solution**

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Customer:
    customer_id: str
    name: str
    email: str | None = None
    tags: list[str] = field(default_factory=list)

    def display_name(self) -> str:
        return f"{self.name} <{self.email}>" if self.email else self.name

**Explanation**

`@dataclass` generates `__init__`, `__repr__`, `__eq__`. `field(default_factory=list)` avoids the shared-mutable-default trap. Attributes are public; that's normal in Python.

**Java habit (don't)**

In [ ]:
class CustomerJavaStyle:
    def __init__(self, customer_id, name):
        self._customer_id = customer_id
        self._name = name
    def getCustomerId(self): return self._customer_id
    def setCustomerId(self, v): self._customer_id = v
    def getName(self): return self._name
    def setName(self, v): self._name = v

**Complexity:** —

**Common mistakes**

- `tags: list = []` → `ValueError: mutable default` (dataclasses refuse it — good!).
- Writing getters/setters 'for encapsulation' before there's any logic.

## Exercise 2 — 🟡 Medium · Setters with validation → @property

**Working solution**

In [ ]:
class Account:
    def __init__(self, account_id: str, email: str, balance: float = 0.0):
        self.account_id = account_id
        self.email = email            # goes through the property setter
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, value: str) -> None:
        value = value.strip().lower()
        if "@" not in value:
            raise ValueError(f"invalid email: {value!r}")
        self._email = value

    def deposit(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        return self._balance

    def withdraw(self, amount: float) -> float:
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount
        return self._balance

**Explanation**

Start with a plain attribute; when validation becomes necessary, swap in a `@property` — callers keep writing `acct.email = ...`,
nothing breaks. That's why Python doesn't need getters/setters up front. The leading underscore means 'internal', by convention.

**Complexity:** O(1).

**Common mistakes**

- Assigning `self._email` directly in `__init__`, bypassing validation.
- Naming the backing field the same as the property → infinite recursion.

## Exercise 3 — 🟡 Medium · Optional chains → None handling

**Working solution**

In [ ]:
def find_customer(customers, customer_id):
    return next((c for c in customers if c.get("customer_id") == customer_id), None)

def customer_city(customers, customer_id, default="UNKNOWN"):
    customer = find_customer(customers, customer_id)
    if customer is None:
        return default
    city = (customer.get("address") or {}).get("city")
    return city.strip() if city and city.strip() else default

**Explanation**

`next(generator, default)` is `stream().filter().findFirst().orElse(null)`. Then plain `if x is None` checks and `or {}` fallbacks.
Use `is None` (identity) for None checks, never `== None`. Type hints say `dict | None` to document the 'Optional'.

**Complexity:** O(n).

**Common mistakes**

- `[c for c in customers if ...][0]` → `IndexError` when missing and scans everything.
- `if not value` when `0` or `''` are valid values.

## Exercise 4 — 🟠 Hard · Builder pattern → frozen dataclass + replace

**Working solution**

In [ ]:
from dataclasses import dataclass, fields, replace

@dataclass(frozen=True)
class JobConfig:
    name: str
    input_path: str
    output_path: str
    batch_size: int = 1000
    max_retries: int = 3
    dry_run: bool = False
    tags: tuple[str, ...] = ()

    def __post_init__(self):
        if not self.name.strip():
            raise ValueError("name is required")
        if self.batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        if not 0 <= self.max_retries <= 10:
            raise ValueError("max_retries must be between 0 and 10")
        if self.input_path == self.output_path:
            raise ValueError("input_path and output_path must differ")

    @classmethod
    def from_dict(cls, data: dict) -> "JobConfig":
        allowed = {f.name for f in fields(cls)}
        unknown = sorted(set(data) - allowed)
        if unknown:
            raise ValueError(f"unknown config keys: {unknown}")
        clean = {k: v for k, v in data.items() if v is not None}
        if "tags" in clean:
            clean["tags"] = tuple(clean["tags"])
        return cls(**clean)

    def with_overrides(self, **changes) -> "JobConfig":
        return replace(self, **changes)

**Explanation**

Keyword arguments with defaults *are* the builder. `frozen=True` gives immutability (and hashability); `replace()` makes modified
copies and re-runs `__post_init__`. `cls(**clean)` unpacks a dict into keyword arguments; `classmethod` is Python's static factory.

**Complexity:** O(fields).

**Common mistakes**

- Silently ignoring unknown keys — typos like `batchsize` then do nothing.
- Mutable `tags: list` on a frozen dataclass (unhashable and mutable).

## Challenge — 🔴 Very Hard · Static factory class → module-level registry

**Working solution**

In [ ]:
import csv
import io
import json

_PARSERS = {}

def register(fmt):
    key = fmt.lower()
    def decorator(func):
        if key in _PARSERS:
            raise ValueError(f"parser already registered for {key!r}")
        _PARSERS[key] = func
        return func
    return decorator

def available_formats():
    return sorted(_PARSERS)

def parse(fmt, text):
    try:
        parser = _PARSERS[fmt.lower()]
    except KeyError:
        raise ValueError(f"unknown format {fmt!r}; available: {available_formats()}") from None
    return parser(text)

@register("csv")
def _parse_csv(text):
    return list(csv.DictReader(io.StringIO(text)))

@register("jsonl")
def _parse_jsonl(text):
    return [json.loads(line) for line in text.splitlines() if line.strip()]

@register("kv")
def _parse_kv(text):
    return [dict(tok.split("=", 1) for tok in line.split()) for line in text.splitlines() if line.strip()]

**Explanation**

A module *is* a singleton namespace — no `private constructor` + `static` needed. Registering functions with a decorator lets new formats be
added without editing a `switch`. This is how Flask routes, pytest fixtures, and Airflow tasks are wired.

**Complexity:** O(1) dispatch.

**Common mistakes**

- A class with only `@staticmethod`s.
- Forgetting to `return func` from the decorator — the decorated name becomes `None`.